In [28]:
import json
import time
import os
import re
from openai import OpenAI
from pathlib import Path

In [ ]:
# new model to test
api_key = ""      # 请替换为你自己的 API key

Client = OpenAI(
    api_key=api_key,
    base_url=""   # 请替换为你自己的 url
)
model_name = "gpt-5"   # 请替换为你自己的 model

In [ ]:
instruction = (
    "Please determine whether there is a non-crash functional bug in the following app operation log, and explain the sequence by analyzing the meaning and result of each recorded operation.\n"
    "output format {\n\t""is_bug"": true or false,\n\t""reason"": brief explain why do you consider it is a bug or it does not has bug\n}"
    "UI info interaction trace:\n"
)

"""
# 读取示例内容
example_file_path = "/Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/script/ICLexamples.txt"
with open(example_file_path, "r", encoding="utf-8") as f:
    example_content = f.read()

# 要插入的位置标志
insert_marker = "UI info interaction trace:\n"

# 插入 example 内容 + 原标志
instruction = instruction.replace(
    insert_marker,
    "\n" + example_content + "\n" + insert_marker
)
"""

# 打印确认
print(instruction)

excluded_ids = {}
max_elements = 51
rate_pause_sec = 2

def get_next_try_filename(folder, prefix, model_name):
    """返回包含下一个 try 编号的完整文件路径"""
    filename_prefix = f"{prefix}_{model_name}_try"
    pattern = re.compile(rf"^{re.escape(filename_prefix)}(\d+)\.json$")

    existing_files = os.listdir(folder)
    existing_tries = [
        int(match.group(1)) for file in existing_files
        if (match := pattern.match(file))
    ]
    next_try = max(existing_tries, default=0) + 1
    return os.path.join(folder, f"{filename_prefix}{next_try}.json")

def process_file(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as f:
        all_data = json.load(f)

    results_json = []
    count = 0

    for item in all_data:
        entry_id = item.get("id")
        if entry_id in excluded_ids:
            continue
        if count >= max_elements:
            break

        try:
            content = item.get("actions_content", "").strip()
            prompt = instruction + content

            response = Client.responses.create(
                model=model_name,
                # temperature=0.7,
                input=[
                    {"role": "user", "content": prompt},
                ],
            )

            raw_output = response.output_text

            results_json.append({
                "id": entry_id,
                "output": raw_output
            })

            print(f"[✓] Processed id={entry_id} ({count+1}/{max_elements})")
            # time.sleep(rate_pause_sec)
            count += 1

        except Exception as e:
            print(f"[✗] Error on id={entry_id}: {e}")
            continue

    with open(output_path, "w", encoding="utf-8") as f_out:
        json.dump(results_json, f_out, ensure_ascii=False, indent=2)

    print(f"\n✅ Saved {count} results to: {output_path}")


Please determine whether there is a non-crash functional bug in the following app operation log, and explain the sequence by analyzing the meaning and result of each recorded operation.
output format {
	is_bug: true or false,
	reason: brief explain why do you consider it is a bug or it does not has bug
}UI info interaction trace:



In [ ]:
# 设置基础参数
model_name = "gpt-5"
folder = ""

# ==== 执行两次：一次处理 buggy 数据，一次处理 bug-free 数据 ====

# Buggy 数据
buggy_input = ""
buggy_output = get_next_try_filename(folder, "testset_buggy", model_name)
process_file(buggy_input, buggy_output)


# Bug-free 数据
bugfree_input = ""
bugfree_output = get_next_try_filename(folder, "testset_bugfree", model_name)
process_file(bugfree_input, bugfree_output)

print(f"Processing buggy output: {buggy_output}")
print(f"Processing bugfree output: {bugfree_output}")


[✓] Processed id=1 (1/62)
[✓] Processed id=2 (2/62)
[✓] Processed id=3 (3/62)
[✓] Processed id=4 (4/62)
[✓] Processed id=6 (5/62)
[✓] Processed id=7 (6/62)
[✓] Processed id=8 (7/62)
[✓] Processed id=9 (8/62)
[✓] Processed id=10 (9/62)
[✓] Processed id=11 (10/62)
[✓] Processed id=12 (11/62)
[✓] Processed id=13 (12/62)
[✓] Processed id=14 (13/62)
[✓] Processed id=15 (14/62)
[✓] Processed id=17 (15/62)
[✓] Processed id=18 (16/62)
[✓] Processed id=19 (17/62)
[✓] Processed id=20 (18/62)
[✓] Processed id=21 (19/62)
[✓] Processed id=22 (20/62)
[✓] Processed id=23 (21/62)
[✓] Processed id=24 (22/62)
[✓] Processed id=25 (23/62)
[✓] Processed id=26 (24/62)
[✓] Processed id=27 (25/62)
[✓] Processed id=28 (26/62)
[✓] Processed id=29 (27/62)
[✓] Processed id=30 (28/62)
[✓] Processed id=31 (29/62)
[✓] Processed id=32 (30/62)
[✓] Processed id=33 (31/62)
[✓] Processed id=34 (32/62)
[✓] Processed id=35 (33/62)
[✓] Processed id=36 (34/62)
[✓] Processed id=38 (35/62)
[✓] Processed id=39 (36/62)
[✓] Proce

In [ ]:
import re
import pandas as pd

# GPT4o to compare
openai_api_key = ""  # 请替换为你自己的 openai API key
Client = OpenAI(
    api_key=openai_api_key,
)

rate_pause_sec = 1
label_csv_path = ""    # ground truth path
# 基于已有 buggy_output 构造 output_compare_csv
output_compare_csv = buggy_output.replace("testset", "compare").replace(".json", ".csv")

print(output_compare_csv)

# 加载数据
with open(buggy_output, "r", encoding="utf-8") as f:
    ai_outputs = {str(item["id"]): item["output"] for item in json.load(f)}

labels_df = pd.read_csv(label_csv_path)
labels_df["id"] = labels_df["id"].astype(str)

/Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/generated/compare_buggy_gpt-5_try3.csv


In [33]:
# prompt模板
prompt_template = """You are given two pieces of information about a potential bug in an app's operation:

1. Official bug label:
- Is Bug: {is_bug}
- Reason: {reason}

2. AI-generated output:
- Output: {output}

Task: Determine if the AI output agrees with the official bug label and reason. Focus on whether the AI's reasoning matches the official reason and whether it correctly reflects the bug/non-bug status.

If they agree:
Start your answer with:
"Yes. They agree."
Then explain briefly why.

If they do NOT agree:
Start your answer with:
"No. They do not agree."
Then explain briefly why.
"""

# 简单提取 is_bug 的兜底函数
def fallback_extract_is_bug_only(text):
    match = re.search(r'"is_bug"\s*:\s*(true|false)', text, re.IGNORECASE)
    if match:
        return match.group(1).lower() == 'true'
    return None

def fallback_extract_is_bug_only(text):
    match = re.search(r'["\']?is_bug["\']?\s*:\s*(true|false)', text, re.IGNORECASE)
    if match:
        return match.group(1).lower() == 'true'
    return None

In [34]:
results = []
model_name = "gpt-4o"

for _, row in labels_df.iterrows():
    record_id = str(row["id"])
    gt_is_bug = bool(row["is_bug"])
    gt_reason = row["reason"]

    ai_raw = ai_outputs.get(record_id)
    if not ai_raw:
        print(f"[!] No AI output for ID={record_id}")
        continue

    ai_is_bug = fallback_extract_is_bug_only(ai_raw)
    if ai_is_bug is None:
        print(f"[!] AI output missing is_bug for ID={record_id}")
        continue

    label = ""
    comment = ""

    if not gt_is_bug and not ai_is_bug:
        label = "TN"

    elif not gt_is_bug and ai_is_bug:
        label = "FP"

    elif gt_is_bug and not ai_is_bug:
        label = "FN"

    elif gt_is_bug and ai_is_bug:
        prompt = prompt_template.format(is_bug=gt_is_bug, reason=gt_reason, output=ai_raw)
        try:
            response = Client.chat.completions.create(
                model=model_name,
                temperature=0.7,
                messages=[{"role": "user", "content": prompt}]
            )
            result_text = response.choices[0].message.content.strip()
            label = "TPC" if result_text.lower().startswith("yes") else "TPW"
            comment = result_text
            time.sleep(rate_pause_sec)
        except Exception as e:
            print(f"[✗] Error during GPT call for ID={record_id}: {e}")
            label = "TPC_or_TPW_error"
            comment = "GPT error"

    results.append({
        "id": record_id,
        "ai_is_bug": ai_is_bug,
        "label": label,
        "comment": comment
    })

# 保存结果
pd.DataFrame(results).to_csv(output_compare_csv, index=False)
print(f"\n✅ Final comparison saved to: {output_compare_csv}")


[!] No AI output for ID=5
[!] No AI output for ID=16
[!] No AI output for ID=37
[!] No AI output for ID=66
[!] No AI output for ID=67
[!] No AI output for ID=68
[!] No AI output for ID=69
[!] No AI output for ID=70
[!] No AI output for ID=71
[!] No AI output for ID=72
[!] No AI output for ID=73
[!] No AI output for ID=74
[!] No AI output for ID=75
[!] No AI output for ID=76
[!] No AI output for ID=77
[!] No AI output for ID=78
[!] No AI output for ID=79
[!] No AI output for ID=80
[!] No AI output for ID=81
[!] No AI output for ID=82
[!] No AI output for ID=83
[!] No AI output for ID=84
[!] No AI output for ID=85
[!] No AI output for ID=86
[!] No AI output for ID=87
[!] No AI output for ID=88
[!] No AI output for ID=89
[!] No AI output for ID=90
[!] No AI output for ID=91
[!] No AI output for ID=92
[!] No AI output for ID=93
[!] No AI output for ID=94
[!] No AI output for ID=95
[!] No AI output for ID=96
[!] No AI output for ID=97
[!] No AI output for ID=98
[!] No AI output for ID=99
[!